In [ ]:
source_json = "{}"
api_token = ""

In [ ]:
# Importaciones y Configuración
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType, StringType, IntegerType
import json
import requests
import uuid
import traceback
import xml.etree.ElementTree as ET

source = json.loads(source_json)
# assert "ingestion" in source, f"el parametro 'sources' no llego al notebook: {source_json[:200]}"

today = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
lookback = source["ingestion"]["lookback_days"]
step = timedelta(days=source["ingestion"]["window_size_days"])

# Funciones

In [ ]:
# Extract
def extract_data(day):
    # Creamos las ventanas de fecha
    ws, we = day, day + step
    req = source["request"]

    if source["fetcher"] == "http_single":
        params = {}
        for k, v in req.get("query_params", {}).items():
            if v is None:
                continue  
            params[k] = v.format(window_start=ws, window_end=we) if isinstance(v, str) and "{window" in v else v

        if source["auth"]["type"] == "query_param":
            params[source["auth"]["param_name"]] = api_token
        
        r = requests.get(req["url"], params=params, timeout=30)
        r.raise_for_status()
        return r.text if source["format"] == "xml" else r.json()
    
    if source["fetcher"] == "http_index_then_file":
        idx = requests.get(req["index_url"], timeout=30).json()[req["index_selector"]]
        candidates = [t for t in sorted(idx) if t <= int(ws.timestamp() * 1000)]
        file_ts = candidates[-1] if candidates else sorted(idx)[0]
        url = req["data_url"].format(file_timestamp=file_ts)
        
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        return r.json()
    
    raise ValueError(f"Endpoint desconocido: {source['fetcher']}")

# Parseo
def parse_data(raw, ws, we):
    if source["parser"] == "entsoe_xml":
        root = ET.fromstring(raw)
        for e in root.iter():
            e.tag = e.tag.split('}')[-1]

        rows = []
        for ts in root.findall(".//TimeSeries"):
            if ts.findtext("contract_MarketAgreement.type") not in (None, "A01"):
                continue  # descarta subastas intradiarias, solo day-ahead

            period = ts.find("Period")
            start = datetime.fromisoformat(period.find("timeInterval/start").text.replace("Z", "+00:00"))
            end = datetime.fromisoformat(period.find("timeInterval/end").text.replace("Z", "+00:00"))
            minutes = int(period.find("resolution").text.replace("PT", "").replace("M", ""))
            total = int((end - start).total_seconds() / 60 / minutes)

            pts = [(int(p.find("position").text), float(p.find("price.amount").text))
                   for p in period.findall("Point")]
            for j, (pos, price) in enumerate(pts):
                hasta = pts[j + 1][0] if j + 1 < len(pts) else total + 1
                for q in range(pos, hasta):  # curveType A03: rellena las posiciones omitidas
                    rows.append((start + timedelta(minutes=minutes * (q - 1)), price))

        return rows

    if source["parser"] == "smard_json":
        return [
            (datetime.fromtimestamp(t / 1000, tz=timezone.utc), float(v))
            for t, v in raw.get("series", [])
            if v is not None and ws <= datetime.fromtimestamp(t / 1000, tz=timezone.utc) < we
        ]

    if source["parser"] == "pse_json":
        return [
            (datetime.fromisoformat(r["dtime_utc"]).replace(tzinfo=timezone.utc), float(r["rce_pln"]))
            for r in raw.get("value", raw if isinstance(raw, list) else [])
        ]

    raise ValueError(f"parser desconocido: {source['parser']}")

# Conersión a EUR
def convert_to_eur(price, currency, day):
    if currency == "EUR":
        return price
    
    d = day
    for _ in range(5):
        try:
            r = requests.get(f"https://api.nbp.pl/api/exchangerates/rates/a/eur/{d:%Y-%m-%d}/?format=json", timeout=10)
            if r.ok:
                rate = r.json()["rates"][0]["mid"]
                return price / rate
        except requests.RequestException:
            pass
        
        d -= timedelta(days=1)
    
    raise RuntimeError("No se encontro tasa NBP en los 5 días anteriores")

## Ejecución

In [ ]:
# Ejecución
all_rows = []
rows_loaded = 0
status = "success"
error = None

try:
    for i in range(lookback, -1, -1):
        day = today - timedelta(days=i)
        raw = extract_data(day)

        for ts, price in parse_data(raw, day, day + step):
            all_rows.append((ts, convert_to_eur(price, source["currency"], day)))

except Exception as e:
    status, error = "error", str(e)
    traceback.print_exc()

if all_rows:
    schema = StructType([
        StructField("timestamp_utc", TimestampType(), False),
        StructField("price_eur", DoubleType(), False)
    ])
    df = (
        spark.createDataFrame(all_rows, schema=schema)
        .withColumn("country_code", F.lit(source["country_code"]))
        .withColumn("source_id", F.lit(source["source_id"]))
        .withColumn("ingested_at", F.current_timestamp())
        .dropDuplicates(["country_code", "timestamp_utc"])
    )
    rows_loaded = df.count()
    df.createOrReplaceTempView("staging")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {source['target_table'].split('.')[0]}")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {source['target_table']} USING DELTA AS SELECT * FROM staging WHERE 1=0")
    spark.sql(f"""
        MERGE INTO {source['target_table']} AS t
        USING staging AS s
        ON t.timestamp_utc = s.timestamp_utc AND t.country_code = s.country_code
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Logs

In [ ]:
window_start_utc = today - timedelta(days=lookback)
window_end_utc = today + step

log_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_id", StringType(), False),
    StructField("window_start_utc", TimestampType(), False),
    StructField("window_end_utc", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("rows_loaded", IntegerType(), False),
    StructField("error_message", StringType(), True),
    StructField("executed_at_utc", TimestampType(), False),
])

spark.createDataFrame([(
    str(uuid.uuid4()), source["source_id"], window_start_utc, window_end_utc,
    status, rows_loaded, error, datetime.now(timezone.utc)
)], schema=log_schema) \
    .write.format("delta").mode("append").saveAsTable("control.run_log")

if status == "error":
    raise RuntimeError(error)
